In [37]:
import pandas as pd
import numpy as np
!pip install xgboost
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
df = pd.read_csv("../../data/kidney.csv")

In [3]:
df.shape

(400, 26)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    object 
 17  wc              295 non-null    obj

In [18]:
num_cols = df.select_dtypes(exclude=['object']).columns

In [19]:
num_cols

Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot',
       'hemo', 'pcv', 'wc', 'rc'],
      dtype='object')

In [20]:
for num in num_cols:
    df[num] = df[num].fillna(df[num].median())

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             400 non-null    float64
 2   bp              400 non-null    float64
 3   sg              400 non-null    float64
 4   al              400 non-null    float64
 5   su              400 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             400 non-null    float64
 11  bu              400 non-null    float64
 12  sc              400 non-null    float64
 13  sod             400 non-null    float64
 14  pot             400 non-null    float64
 15  hemo            400 non-null    float64
 16  pcv             400 non-null    float64
 17  wc              400 non-null    flo

In [23]:
cat_cols = df.select_dtypes(include=['object']).columns

In [24]:
cat_cols

Index(['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane',
       'classification'],
      dtype='object')

In [29]:
for col in cat_cols:
    print(col,"=====",df[col].unique())

rbc ===== [1. 0.]
pc ===== [1. 0.]
pcc ===== [0. 1.]
ba ===== [0. 1.]
htn ===== [1. 0.]
dm ===== [1. 0.]
cad ===== [0. 1.]
appet ===== [1. 0.]
pe ===== [0. 1.]
ane ===== [0. 1.]
classification ===== [1 0]


In [12]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [13]:
df.replace('?', np.nan, inplace=True)

In [16]:
cat_num = ['pcv','wc','rc']
for col in cat_num:
    df[col] = pd.to_numeric(df[col],errors='coerce')

In [26]:
df['rbc'] = df['rbc'].map({'normal': 1, 'abnormal': 0})
df['pc'] = df['pc'].map({'normal': 1, 'abnormal': 0})
df['pcc'] = df['pcc'].map({'present': 1, 'notpresent': 0})
df['ba'] = df['ba'].map({'present': 1, 'notpresent': 0})
df['htn'] = df['htn'].map({'yes': 1, 'no': 0})
df['dm'] = df['dm'].map({'yes': 1, 'no': 0})
df['cad'] = df['cad'].map({'yes': 1, 'no': 0})
df['appet'] = df['appet'].map({'good': 1, 'poor': 0})
df['pe'] = df['pe'].map({'yes': 1, 'no': 0})
df['ane'] = df['ane'].map({'yes': 1, 'no': 0})
df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})

In [28]:
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\chinm\AppData\Local\Temp\ipykernel_24056\1925696021.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
C:\Users\chinm\AppData\Local\Temp\ipykernel_24056\1925696021.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             400 non-null    float64
 2   bp              400 non-null    float64
 3   sg              400 non-null    float64
 4   al              400 non-null    float64
 5   su              400 non-null    float64
 6   rbc             400 non-null    float64
 7   pc              400 non-null    float64
 8   pcc             400 non-null    float64
 9   ba              400 non-null    float64
 10  bgr             400 non-null    float64
 11  bu              400 non-null    float64
 12  sc              400 non-null    float64
 13  sod             400 non-null    float64
 14  pot             400 non-null    float64
 15  hemo            400 non-null    float64
 16  pcv             400 non-null    float64
 17  wc              400 non-null    flo

In [31]:
df = df.drop_duplicates()

In [32]:
df.shape

(400, 26)

In [33]:
df.columns

Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr',
       'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [61]:
X = df[['age', 'bp', 
             'htn', 'dm', 
       'appet']]

y = df['classification']

In [62]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [63]:
rf = RandomForestClassifier(n_estimators=200,max_depth=8,random_state=42)
xg = XGBClassifier(random_state = 42,
    eval_metric = "logloss",
    use_label_encoder = False)

In [64]:
rf.fit(X_train,y_train)
xg.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)
y_pred_xg = xg.predict(X_test)
rf_accuracy = accuracy_score(y_test,y_pred_rf)
xg_accuracy = accuracy_score(y_test,y_pred_xg)
print("Rf accuracy", rf_accuracy)
print("xg accuracy", xg_accuracy)

Rf accuracy 0.9125
xg accuracy 0.9125


In [65]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(by='importance', ascending=False)
feature_importance

,feature,importance
2,htn,0.276828
3,dm,0.259778
0,age,0.175783
1,bp,0.152708
4,appet,0.134902


In [66]:
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.80      1.00      0.89        28
           1       1.00      0.87      0.93        52

    accuracy                           0.91        80
   macro avg       0.90      0.93      0.91        80
weighted avg       0.93      0.91      0.91        80



In [67]:
from sklearn.model_selection import RandomizedSearchCV
param_dist_rf = {
    "n_estimators": np.arange(100, 500, 50),
    "max_depth": [None, 4, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

param_dist_xgb = {
    "n_estimators": np.arange(100, 400, 50),
    "max_depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3]
}
rf = RandomForestClassifier(class_weight='balanced',random_state=42)
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=20,              # number of combinations to try
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [None, 4, 6, 8, 10],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': array([100, 150, 200, 250, 300, 350, 400, 450])},
                   random_state=42, verbose=2)

In [68]:
print("Best Params:", rf_random.best_params_)
print("Best Score:", rf_random.best_score_)

best_rf = rf_random.best_estimator_

Best Params: {'n_estimators': np.int64(400), 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 4}
Best Score: 0.909375


In [69]:
xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=25,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_cons...
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.2, 0.3],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 4, 5, 6, 7],
                                        'n_estimators': array([100, 150, 200, 250, 300, 350]),
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, verbose=2)

In [70]:
print("Best Params:", xgb_random.best_params_)
print("Best Score:", xgb_random.best_score_)

best_xgb = xgb_random.best_estimator_

Best Params: {'subsample': 0.9, 'n_estimators': np.int64(350), 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0.3, 'colsample_bytree': 1.0}
Best Score: 0.909375


In [71]:
from sklearn.metrics import accuracy_score

rf_pred = best_rf.predict(X_test)
xgb_pred = best_xgb.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("XGB Accuracy:", accuracy_score(y_test, xgb_pred))

RF Accuracy: 0.9
XGB Accuracy: 0.9


In [72]:
from sklearn.metrics import classification_report
print(classification_report(y_test, rf_pred))
print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.78      1.00      0.88        28
           1       1.00      0.85      0.92        52

    accuracy                           0.90        80
   macro avg       0.89      0.92      0.90        80
weighted avg       0.92      0.90      0.90        80

              precision    recall  f1-score   support

           0       0.78      1.00      0.88        28
           1       1.00      0.85      0.92        52

    accuracy                           0.90        80
   macro avg       0.89      0.92      0.90        80
weighted avg       0.92      0.90      0.90        80

